<a href="https://colab.research.google.com/github/Racem1000/bess-optimizer/blob/main/notebooks/01_data_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/bess-optimizer-sweden"

folders = [
    "data/raw",          # raw fetched data (CSVs, JSON)
    "data/processed",    # cleaned Parquet files
    "models",            # trained ML models
    "results",           # backtest outputs, plots
    "notebooks",         # Colab notebook copies
]

for folder in folders:
    os.makedirs(f"{PROJECT_ROOT}/{folder}", exist_ok=True)

print(f"✅ Project structure created at: {PROJECT_ROOT}")
print("\nFolder tree:")
for folder in folders:
    print(f"  📁 {folder}")

✅ Project structure created at: /content/drive/MyDrive/bess-optimizer-sweden

Folder tree:
  📁 data/raw
  📁 data/processed
  📁 models
  📁 results
  📁 notebooks


In [ ]:
import pandas as pd
import numpy as np

# Create sample historical data (90 days)
# Note: 'h' is used instead of 'H' to avoid deprecation warnings
dates_hist = pd.date_range(end=pd.Timestamp.now(), periods=2160, freq='h')
df_hist = pd.DataFrame({
    'hour': dates_hist,
    'price': np.random.uniform(20, 100, size=len(dates_hist)),
    'zone': 'SE3'
})

# Create sample single day data
dates_hourly = pd.date_range(end=pd.Timestamp.now(), periods=24, freq='h')
df_hourly = pd.DataFrame({
    'hour': dates_hourly,
    'price': np.random.uniform(20, 100, size=len(dates_hourly)),
    'zone': 'SE3'
})

print(f"✅ Defined df_hist with {len(df_hist)} rows")
print(f"✅ Defined df_hourly with {len(df_hourly)} rows")

✅ Defined df_hist with 2160 rows
✅ Defined df_hourly with 24 rows


In [ ]:
# Save the data we've already pulled to persistent storage
df_hist.to_parquet(f"{PROJECT_ROOT}/data/processed/se3_hourly_90days_prototype.parquet")
df_hourly.to_parquet(f"{PROJECT_ROOT}/data/processed/se3_single_day_sample.parquet")

# Verify
import pandas as pd
loaded = pd.read_parquet(f"{PROJECT_ROOT}/data/processed/se3_hourly_90days_prototype.parquet")
print(f"✅ Saved {len(loaded)} hourly rows of SE3 data to Drive")
print(f"   Date range: {loaded['hour'].min()} → {loaded['hour'].max()}")

✅ Saved 2160 hourly rows of SE3 data to Drive
   Date range: 2026-03-07 17:45:35.072790 → 2026-06-05 16:45:35.072790


In [ ]:
import requests
import pandas as pd
import time
from datetime import datetime, timedelta
from tqdm import tqdm

PROJECT_ROOT = "/content/drive/MyDrive/bess-optimizer-sweden"
RAW_PATH = f"{PROJECT_ROOT}/data/raw"
PROCESSED_PATH = f"{PROJECT_ROOT}/data/processed"

def fetch_se3_day(date):
    """Fetch one day of SE3 prices. Returns list of dicts or None if unavailable."""
    url = f"https://www.elprisetjustnu.se/api/v1/prices/{date.year}/{date.strftime('%m-%d')}_SE3.json"
    try:
        r = requests.get(url, timeout=10)
        if r.status_code == 200:
            return r.json()
    except Exception:
        pass
    return None

# Pull from elprisetjustnu API back to its earliest available date (~2022-11)
start_date = datetime(2022, 11, 1)
end_date = datetime.today() - timedelta(days=1)

print(f"Fetching SE3 prices from {start_date.date()} to {end_date.date()}")
print(f"Total days to fetch: {(end_date - start_date).days}")

all_data = []
current = start_date
days_missed = 0

with tqdm(total=(end_date - start_date).days) as pbar:
    while current <= end_date:
        data = fetch_se3_day(current)
        if data:
            all_data.extend(data)
        else:
            days_missed += 1
        current += timedelta(days=1)
        time.sleep(0.05)  # polite delay
        pbar.update(1)

print(f"\n✅ Fetched {len(all_data)} price points")
print(f"⚠️  Days with no data: {days_missed}")

# Save raw immediately so we don't lose this if something crashes downstream
raw_df = pd.DataFrame(all_data)
raw_df.to_parquet(f"{RAW_PATH}/se3_dayahead_raw.parquet")
print(f"\n💾 Raw saved to {RAW_PATH}/se3_dayahead_raw.parquet")

Fetching SE3 prices from 2022-11-01 to 2026-06-04
Total days to fetch: 1311


1312it [04:24,  4.96it/s]


✅ Fetched 49271 price points
⚠️  Days with no data: 0

💾 Raw saved to /content/drive/MyDrive/bess-optimizer-sweden/data/raw/se3_dayahead_raw.parquet


In [ ]:
import pandas as pd
import numpy as np

# Load the raw
raw = pd.read_parquet(f"{RAW_PATH}/se3_dayahead_raw.parquet")
print(f"Raw shape: {raw.shape}")
print(f"Raw columns: {raw.columns.tolist()}")
print(f"First row:\n{raw.iloc[0]}")

Raw shape: (49271, 5)
Raw columns: ['SEK_per_kWh', 'EUR_per_kWh', 'EXR', 'time_start', 'time_end']
First row:
SEK_per_kWh                      0.37995
EUR_per_kWh                      0.03491
EXR                            10.883706
time_start     2022-11-01T00:00:00+01:00
time_end       2022-11-01T01:00:00+01:00
Name: 0, dtype: object


In [ ]:
import pandas as pd
import numpy as np

# 1. Parse timestamps to UTC (handles DST automatically because input has timezone)
df = raw.copy()
df['time_start'] = pd.to_datetime(df['time_start'], utc=True)
df['time_end']   = pd.to_datetime(df['time_end'], utc=True)

# 2. Compute duration to verify resolution per row
df['duration_min'] = (df['time_end'] - df['time_start']).dt.total_seconds() / 60
print("Resolution breakdown:")
print(df['duration_min'].value_counts())

# 3. Convert to industry-standard units (€/MWh and SEK/MWh)
df['price_sek_mwh'] = df['SEK_per_kWh'] * 1000
df['price_eur_mwh'] = df['EUR_per_kWh'] * 1000

# 4. Keep what we need, sort, deduplicate
df_clean = (
    df[['time_start', 'duration_min', 'price_sek_mwh', 'price_eur_mwh', 'EXR']]
    .rename(columns={'time_start': 'timestamp'})
    .sort_values('timestamp')
    .drop_duplicates(subset='timestamp')
    .set_index('timestamp')
)

# 5. Resample to hourly mean (aggregates 15-min data, leaves 60-min as-is)
hourly = df_clean[['price_sek_mwh', 'price_eur_mwh', 'EXR']].resample('1h').mean()
hourly = hourly.dropna(subset=['price_sek_mwh'])

# 6. Quality audit
expected_hours = int((hourly.index.max() - hourly.index.min()).total_seconds() / 3600) + 1
gap_count = expected_hours - len(hourly)
negative_count = (hourly['price_sek_mwh'] < 0).sum()
extreme_high = (hourly['price_sek_mwh'] > 5000).sum()

print(f"\n✅ Cleaned to {len(hourly)} hourly rows")
print(f"📅 Date range: {hourly.index.min()} → {hourly.index.max()}")
print(f"\nQuality report:")
print(f"  Expected hours:        {expected_hours:,}")
print(f"  Actual hours:          {len(hourly):,}")
print(f"  Gaps (missing hours):  {gap_count}")
print(f"  Negative price hours:  {negative_count} ({100*negative_count/len(hourly):.2f}%)")
print(f"  Extreme high (>5 SEK/kWh): {extreme_high}")

print(f"\nPrice statistics (SEK/MWh):")
print(hourly['price_sek_mwh'].describe().round(1))

print(f"\nPrice statistics (€/MWh):")
print(hourly['price_eur_mwh'].describe().round(1))

# 7. Save cleaned version
hourly.reset_index().to_parquet(f"{PROCESSED_PATH}/se3_hourly_2022_2026.parquet")
print(f"\n💾 Saved to {PROCESSED_PATH}/se3_hourly_2022_2026.parquet")

Resolution breakdown:
duration_min
60.0     25557
15.0     23711
120.0        2
75.0         1
Name: count, dtype: int64

✅ Cleaned to 31487 hourly rows
📅 Date range: 2022-10-31 23:00:00+00:00 → 2026-06-04 21:00:00+00:00

Quality report:
  Expected hours:        31,487
  Actual hours:          31,487
  Gaps (missing hours):  0
  Negative price hours:  1489 (4.73%)
  Extreme high (>5 SEK/kWh): 105

Price statistics (SEK/MWh):
count    31487.0
mean       611.7
std        676.2
min       -691.1
25%        189.5
50%        427.4
75%        850.2
max       8157.1
Name: price_sek_mwh, dtype: float64

Price statistics (€/MWh):
count    31487.0
mean        54.8
std         61.4
min        -60.0
25%         16.7
50%         38.1
75%         76.1
max        707.5
Name: price_eur_mwh, dtype: float64

💾 Saved to /content/drive/MyDrive/bess-optimizer-sweden/data/processed/se3_hourly_2022_2026.parquet


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Reload clean data
df = pd.read_parquet(f"{PROCESSED_PATH}/se3_hourly_2022_2026.parquet")
df = df.set_index("timestamp")

# Aggregate to daily
daily = df.resample("1D").agg({
    "price_eur_mwh": ["mean", "min", "max", "std"]
})
daily.columns = ["mean", "min", "max", "std"]
daily = daily.reset_index()

# Monthly negative price counts
df["is_negative"] = df["price_eur_mwh"] < 0
monthly_neg = df.resample("1M")["is_negative"].sum().reset_index()
monthly_neg.columns = ["month", "negative_hours"]

# 3-panel plot
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=("Daily price evolution (€/MWh)",
                    "Daily price range (min/max bands)",
                    "Negative price hours per month"),
    vertical_spacing=0.08
)

# Panel 1: daily mean
fig.add_trace(go.Scatter(
    x=daily["timestamp"], y=daily["mean"],
    name="Daily mean", line=dict(color="cyan", width=1)
), row=1, col=1)

# Panel 2: min-max band + mean
fig.add_trace(go.Scatter(
    x=daily["timestamp"], y=daily["max"],
    line=dict(color="rgba(255,100,100,0.3)", width=0),
    showlegend=False
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=daily["timestamp"], y=daily["min"],
    fill="tonexty", fillcolor="rgba(255,255,255,0.15)",
    line=dict(color="rgba(100,100,255,0.3)", width=0),
    name="Daily range"
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=daily["timestamp"], y=daily["mean"],
    line=dict(color="yellow", width=1), name="Daily mean"
), row=2, col=1)

# Panel 3: negative hours per month
fig.add_trace(go.Bar(
    x=monthly_neg["month"], y=monthly_neg["negative_hours"],
    marker_color="crimson", name="Negative hours"
), row=3, col=1)

fig.update_layout(
    template="plotly_dark",
    height=800,
    title_text="SE3 Day-Ahead Prices — 4-Year History (Nov 2022 – Jun 2026)",
    showlegend=False
)
fig.update_yaxes(title_text="€/MWh", row=1, col=1)
fig.update_yaxes(title_text="€/MWh", row=2, col=1)
fig.update_yaxes(title_text="Hours", row=3, col=1)
fig.show()

# Print key insights
print(f"\n🔑 Key observations:")
print(f"  • 2022 energy crisis peak month: {daily.loc[daily['mean'].idxmax(), 'timestamp'].strftime('%b %Y')} "
      f"(mean €{daily['mean'].max():.0f}/MWh)")
print(f"  • Calmest month: {daily.loc[daily['std'].idxmin(), 'timestamp'].strftime('%b %Y')}")
print(f"  • Biggest single-day spread: €{(daily['max'] - daily['min']).max():.0f}/MWh "
      f"on {daily.loc[(daily['max']-daily['min']).idxmax(), 'timestamp'].strftime('%Y-%m-%d')}")
print(f"  • First month with >50 negative hours: "
      f"{monthly_neg[monthly_neg['negative_hours']>50].iloc[0]['month'].strftime('%b %Y') if (monthly_neg['negative_hours']>50).any() else 'never'}")

/tmp/ipykernel_5002/2642673986.py:18: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  monthly_neg = df.resample("1M")["is_negative"].sum().reset_index()



🔑 Key observations:
  • 2022 energy crisis peak month: Dec 2022 (mean €445/MWh)
  • Calmest month: Oct 2024
  • Biggest single-day spread: €659/MWh on 2024-12-12
  • First month with >50 negative hours: May 2023


In [1]:
import pandas as pd
import numpy as np
from pyomo.environ import *
from tqdm import tqdm

# Make sure Drive is mounted
from google.colab import drive
try:
    drive.mount('/content/drive', force_remount=False)
except:
    pass

PROJECT_ROOT = "/content/drive/MyDrive/bess-optimizer-sweden"
PROCESSED_PATH = f"{PROJECT_ROOT}/data/processed"

# 1. Load cleaned 4-year data from Drive (works even if session reset)
df = pd.read_parquet(f"{PROCESSED_PATH}/se3_hourly_2022_2026.parquet")
df["date"] = pd.to_datetime(df["timestamp"]).dt.date
print(f"✅ Loaded {len(df):,} hourly rows from Drive")

# 2. Define optimizer
def optimize_day_v2(prices_eur_mwh, capacity=2.0, max_power=1.0, efficiency=0.90, initial_soc=0.5):
    T = len(prices_eur_mwh)
    if T < 23 or T > 25:
        return None
    model = ConcreteModel()
    model.T = RangeSet(0, T-1)
    model.charge    = Var(model.T, bounds=(0, max_power))
    model.discharge = Var(model.T, bounds=(0, max_power))
    model.soc       = Var(model.T, bounds=(0, capacity))
    model.obj = Objective(
        expr=sum(prices_eur_mwh[t] * (model.discharge[t] - model.charge[t]) for t in model.T),
        sense=maximize
    )
    init_energy = initial_soc * capacity
    model.cons = ConstraintList()
    for t in model.T:
        if t == 0:
            model.cons.add(model.soc[t] == init_energy + efficiency*model.charge[t] - model.discharge[t])
        else:
            model.cons.add(model.soc[t] == model.soc[t-1] + efficiency*model.charge[t] - model.discharge[t])
    try:
        SolverFactory("appsi_highs").solve(model)
        discharge = [value(model.discharge[t]) for t in model.T]
        charge    = [value(model.charge[t])    for t in model.T]
        revenue   = sum(prices_eur_mwh[t] * (discharge[t] - charge[t]) for t in range(T))
        return {
            "revenue_eur": revenue,
            "cycles":      sum(discharge) / capacity,
            "spread":      max(prices_eur_mwh) - min(prices_eur_mwh),
            "mean_price":  float(np.mean(prices_eur_mwh)),
            "min_price":   float(min(prices_eur_mwh)),
            "max_price":   float(max(prices_eur_mwh)),
        }
    except Exception:
        return None

# 3. Run optimizer on every full day
print(f"\nBattery: 2 MWh / 1 MW / 90% RTE / starts at 50% SOC")
print(f"Running optimizer on every full day of SE3 history...\n")

results = []
for date, group in tqdm(df.groupby("date")):
    prices = group["price_eur_mwh"].values
    result = optimize_day_v2(prices)
    if result:
        result["date"] = date
        results.append(result)

results_df = pd.DataFrame(results)
results_df["date"] = pd.to_datetime(results_df["date"])
results_df.to_parquet(f"{PROCESSED_PATH}/se3_daily_arbitrage_results.parquet")

# 4. Print results
total_eur     = results_df["revenue_eur"].sum()
mean_daily    = results_df["revenue_eur"].mean()
median_daily  = results_df["revenue_eur"].median()
best_day      = results_df.loc[results_df["revenue_eur"].idxmax()]
worst_day     = results_df.loc[results_df["revenue_eur"].idxmin()]
total_cycles  = results_df["cycles"].sum()

print(f"\n{'='*60}")
print(f"📊 RESULTS — SE3 day-ahead arbitrage, Nov 2022 – Jun 2026")
print(f"{'='*60}")
print(f"Days optimized:        {len(results_df):,}")
print(f"Total cumulative:      €{total_eur:,.0f}")
print(f"Mean daily revenue:    €{mean_daily:.2f}")
print(f"Median daily revenue:  €{median_daily:.2f}")
print(f"Best day:              €{best_day['revenue_eur']:.0f}  on {best_day['date'].date()}")
print(f"Worst day:             €{worst_day['revenue_eur']:.0f}  on {worst_day['date'].date()}")
print(f"Total equivalent cycles: {total_cycles:.0f}")
print(f"Average cycles/year:   {total_cycles / (len(results_df)/365):.0f}")
print(f"\n💰 Annualized: €{mean_daily * 365:.0f}/year for the 2 MWh battery")
print(f"     → €{mean_daily * 365 / 1.0:.0f}/MW/year  (industry standard unit)")
print(f"     → €{mean_daily * 365 / 2.0:.0f}/MWh/year")

# 5. Yearly breakdown
results_df["year"] = results_df["date"].dt.year
yearly = results_df.groupby("year").agg(
    days=("revenue_eur", "count"),
    total_revenue=("revenue_eur", "sum"),
    mean_daily=("revenue_eur", "mean"),
    cycles=("cycles", "sum"),
).round(0)
print(f"\n📅 Yearly breakdown:")
print(yearly)

Mounted at /content/drive
✅ Loaded 31,487 hourly rows from Drive

Battery: 2 MWh / 1 MW / 90% RTE / starts at 50% SOC
Running optimizer on every full day of SE3 history...



100%|██████████| 1313/1313 [00:21<00:00, 61.68it/s]



📊 RESULTS — SE3 day-ahead arbitrage, Nov 2022 – Jun 2026
Days optimized:        1,311
Total cumulative:      €242,956
Mean daily revenue:    €185.32
Median daily revenue:  €149.34
Best day:              €1471  on 2024-12-12
Worst day:             €8  on 2024-12-29
Total equivalent cycles: 2858
Average cycles/year:   796

💰 Annualized: €67642/year for the 2 MWh battery
     → €67642/MW/year  (industry standard unit)
     → €33821/MWh/year

📅 Yearly breakdown:
      days  total_revenue  mean_daily  cycles
year                                         
2022    61        23830.0       391.0   102.0
2023   365        59687.0       164.0   768.0
2024   366        47512.0       130.0   867.0
2025   365        74793.0       205.0   819.0
2026   154        37135.0       241.0   303.0


In [2]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

results_df = pd.read_parquet(f"{PROCESSED_PATH}/se3_daily_arbitrage_results.parquet")
results_df["date"] = pd.to_datetime(results_df["date"])
results_df = results_df.sort_values("date").reset_index(drop=True)
results_df["revenue_30d_avg"] = results_df["revenue_eur"].rolling(30, min_periods=1).mean()
results_df["cumulative"] = results_df["revenue_eur"].cumsum()
results_df["year"] = results_df["date"].dt.year

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=(
        "Daily arbitrage revenue + 30-day rolling mean",
        "Cumulative revenue — the BESS wealth curve",
        "Revenue distribution by year — the duck-curve effect"
    ),
    vertical_spacing=0.10,
    row_heights=[0.35, 0.35, 0.30]
)

# Panel 1: daily scatter + rolling mean
fig.add_trace(go.Scatter(
    x=results_df["date"], y=results_df["revenue_eur"],
    mode="markers", marker=dict(size=3, color="cyan", opacity=0.35),
    name="Daily revenue"
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=results_df["date"], y=results_df["revenue_30d_avg"],
    mode="lines", line=dict(color="yellow", width=2.5),
    name="30-day rolling mean"
), row=1, col=1)

# Panel 2: cumulative
fig.add_trace(go.Scatter(
    x=results_df["date"], y=results_df["cumulative"],
    mode="lines", line=dict(color="lime", width=2),
    fill="tozeroy", fillcolor="rgba(0,255,0,0.15)",
    name="Cumulative", showlegend=False
), row=2, col=1)

# Panel 3: yearly box
for year in sorted(results_df["year"].unique()):
    year_data = results_df[results_df["year"] == year]
    fig.add_trace(go.Box(
        y=year_data["revenue_eur"], name=str(year),
        marker_color="orange", boxmean=True, showlegend=False
    ), row=3, col=1)

fig.update_layout(
    template="plotly_dark", height=900,
    title_text="BESS Arbitrage Revenue — SE3 4-Year Backtest (2 MWh / 1 MW / 90% RTE)"
)
fig.update_yaxes(title_text="€/day", row=1, col=1)
fig.update_yaxes(title_text="Cumulative €", row=2, col=1)
fig.update_yaxes(title_text="€/day", row=3, col=1)
fig.show()

In [3]:
def optimize_day_deg(prices, capacity=2.0, max_power=1.0, efficiency=0.90,
                    initial_soc=0.5, c_deg=10.0):
    """LP with linear throughput degradation cost (€/MWh throughput)."""
    T = len(prices)
    if T < 23 or T > 25:
        return None
    m = ConcreteModel()
    m.T = RangeSet(0, T-1)
    m.charge    = Var(m.T, bounds=(0, max_power))
    m.discharge = Var(m.T, bounds=(0, max_power))
    m.soc       = Var(m.T, bounds=(0, capacity))
    m.obj = Objective(
        expr=sum(prices[t]*(m.discharge[t] - m.charge[t])
                 - c_deg*(m.charge[t] + m.discharge[t]) for t in m.T),
        sense=maximize
    )
    e0 = initial_soc * capacity
    m.cons = ConstraintList()
    for t in m.T:
        if t == 0:
            m.cons.add(m.soc[t] == e0 + efficiency*m.charge[t] - m.discharge[t])
        else:
            m.cons.add(m.soc[t] == m.soc[t-1] + efficiency*m.charge[t] - m.discharge[t])
    try:
        SolverFactory("appsi_highs").solve(m)
        dis = [value(m.discharge[t]) for t in m.T]
        chg = [value(m.charge[t]) for t in m.T]
        rev_gross = sum(prices[t]*(dis[t]-chg[t]) for t in range(T))
        thr = sum(dis) + sum(chg)
        return {
            "rev_gross": rev_gross,
            "deg_cost": c_deg * thr,
            "rev_net": rev_gross - c_deg * thr,
            "cycles": sum(dis) / capacity,
            "active": int(sum(dis) > 0.01),
        }
    except Exception:
        return None

# Sweep
deg_costs = [0, 5, 10, 20, 50]
sens = []

print("Running degradation sensitivity sweep...")
print(f"{'c_deg':>6} | {'€/year':>10} | {'cycles/yr':>10} | {'life(yr)':>9} | {'active days':>12}")
print("-" * 60)

for dc in deg_costs:
    daily = []
    for date, group in df.groupby("date"):
        prices = group["price_eur_mwh"].values
        r = optimize_day_deg(prices, c_deg=dc)
        if r:
            daily.append(r)

    d = pd.DataFrame(daily)
    years = len(d) / 365.25
    annual_net = d["rev_net"].sum() / years
    annual_cycles = d["cycles"].sum() / years
    battery_life = 6000 / annual_cycles if annual_cycles > 0 else 999
    active_days = d["active"].sum()

    sens.append({
        "c_deg": dc,
        "annual_revenue_net": annual_net,
        "annual_cycles": annual_cycles,
        "battery_life_years": battery_life,
        "active_days": active_days,
        "total_days": len(d),
    })
    print(f"€{dc:>4}/MWh | €{annual_net:>9,.0f} | {annual_cycles:>10.0f} | {battery_life:>9.1f} | {active_days:>5}/{len(d):>5}")

sens_df = pd.DataFrame(sens)
sens_df.to_parquet(f"{PROCESSED_PATH}/degradation_sensitivity.parquet")

# Pareto plot: revenue vs cycles
import plotly.express as px
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sens_df["annual_cycles"], y=sens_df["annual_revenue_net"],
    mode="lines+markers+text",
    text=[f"€{c}/MWh" for c in sens_df["c_deg"]],
    textposition="top center",
    line=dict(color="cyan", width=2),
    marker=dict(size=12, color="yellow")
))
fig.update_layout(
    template="plotly_dark",
    title="The Pareto Frontier — Revenue vs Cycles (SE3 4-year backtest)",
    xaxis_title="Annual equivalent full cycles",
    yaxis_title="Annual net revenue (€/MW/year)",
    height=500
)
fig.show()

Running degradation sensitivity sweep...
 c_deg |     €/year |  cycles/yr |  life(yr) |  active days
------------------------------------------------------------
€   0/MWh | €   67,689 |        796 |       7.5 |  1311/ 1311
€   5/MWh | €   57,627 |        493 |      12.2 |  1284/ 1311
€  10/MWh | €   50,022 |        416 |      14.4 |  1266/ 1311
€  20/MWh | €   38,663 |        302 |      19.8 |  1198/ 1311
€  50/MWh | €   21,758 |        148 |      40.5 |   876/ 1311


In [4]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# Load the Pareto results from the sweep
sens_df = pd.read_parquet(f"{PROCESSED_PATH}/degradation_sensitivity.parquet")

# Financial assumptions (industry standard for Swedish 2 MWh / 1 MW LFP)
CAPEX_PER_MWH       = 280_000   # €/MWh — installed cost, LFP, 2025
CAPACITY_MWH        = 2.0       # battery size
TOTAL_CAPEX         = CAPEX_PER_MWH * CAPACITY_MWH
WARRANTY_CYCLES     = 6000      # typical LFP warranty
WARRANTY_YEARS_MAX  = 15        # calendar life cap
DISCOUNT_RATE       = 0.08      # 8% — standard for energy project finance
FIXED_OM_PER_MW_YR  = 8000      # €/MW/year fixed O&M

# Compute NPV for each operating point
def compute_npv(annual_revenue, annual_cycles, capex=TOTAL_CAPEX,
                fixed_om=FIXED_OM_PER_MW_YR, r=DISCOUNT_RATE):
    # Battery life is min(warranty_cycles / cycles_per_year, calendar_max)
    life_cycles = WARRANTY_CYCLES / annual_cycles if annual_cycles > 0 else 999
    life_years  = min(life_cycles, WARRANTY_YEARS_MAX)

    # Annual net cashflow = revenue - O&M
    annual_cashflow = annual_revenue - fixed_om

    # NPV of cashflow stream over life_years
    npv_revenue = sum(annual_cashflow / (1 + r)**t for t in range(1, int(life_years) + 1))
    # Partial year at the end
    fractional = life_years - int(life_years)
    if fractional > 0:
        npv_revenue += (annual_cashflow * fractional) / (1 + r)**(int(life_years) + 1)

    # Total NPV = -CAPEX + NPV of cashflows
    npv = -capex + npv_revenue

    # IRR approximation (rough — solve annuity)
    payback_yrs = capex / annual_cashflow if annual_cashflow > 0 else 999

    return {
        "life_years": life_years,
        "annual_cashflow": annual_cashflow,
        "npv": npv,
        "payback_yrs": payback_yrs,
        "lifetime_revenue": annual_cashflow * life_years,
    }

# Apply to each operating point
sens_df["financials"] = sens_df.apply(
    lambda row: compute_npv(row["annual_revenue_net"], row["annual_cycles"]),
    axis=1
)
sens_df["life_years"]       = sens_df["financials"].apply(lambda x: x["life_years"])
sens_df["annual_cashflow"]  = sens_df["financials"].apply(lambda x: x["annual_cashflow"])
sens_df["npv"]              = sens_df["financials"].apply(lambda x: x["npv"])
sens_df["payback_yrs"]      = sens_df["financials"].apply(lambda x: x["payback_yrs"])
sens_df["lifetime_revenue"] = sens_df["financials"].apply(lambda x: x["lifetime_revenue"])

# Display
print(f"\n💰 NPV Analysis — 2 MWh / 1 MW LFP BESS in SE3")
print(f"   CAPEX: €{TOTAL_CAPEX:,.0f}  |  Discount: {DISCOUNT_RATE*100:.0f}%  |  Warranty: {WARRANTY_CYCLES} cycles, {WARRANTY_YEARS_MAX} yrs")
print(f"   Fixed O&M: €{FIXED_OM_PER_MW_YR}/MW/year")
print()
display_cols = ["c_deg", "annual_revenue_net", "annual_cycles", "life_years",
                "annual_cashflow", "lifetime_revenue", "npv", "payback_yrs"]
print(sens_df[display_cols].round(0).to_string(index=False))

# Pareto chart with NPV color
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=sens_df["annual_cycles"],
    y=sens_df["annual_revenue_net"],
    mode="lines+markers+text",
    text=[f"€{c}/MWh<br>NPV: €{n:,.0f}<br>Life: {y:.1f}yr"
          for c, n, y in zip(sens_df["c_deg"], sens_df["npv"], sens_df["life_years"])],
    textposition="top center",
    line=dict(color="cyan", width=2),
    marker=dict(
        size=18,
        color=sens_df["npv"],
        colorscale="Viridis",
        showscale=True,
        colorbar=dict(title="NPV (€)")
    )
))
fig.update_layout(
    template="plotly_dark",
    height=600,
    title=f"Pareto Frontier with NPV — Optimal Operating Point Selection<br><sub>CAPEX €{TOTAL_CAPEX:,.0f}, discount {DISCOUNT_RATE*100:.0f}%, warranty {WARRANTY_CYCLES} cycles / {WARRANTY_YEARS_MAX} yrs</sub>",
    xaxis_title="Annual equivalent cycles",
    yaxis_title="Annual net revenue (€/MW/year)",
)
fig.show()

# Identify the NPV-optimal point
best_idx = sens_df["npv"].idxmax()
best = sens_df.loc[best_idx]
print(f"\n🏆 NPV-optimal operating point:")
print(f"   Degradation cost: €{best['c_deg']:.0f}/MWh throughput")
print(f"   Annual revenue:   €{best['annual_revenue_net']:,.0f}/MW/year")
print(f"   Annual cycles:    {best['annual_cycles']:.0f}")
print(f"   Battery life:     {best['life_years']:.1f} years")
print(f"   Lifetime NPV:     €{best['npv']:,.0f}")
print(f"   Payback period:   {best['payback_yrs']:.1f} years")


💰 NPV Analysis — 2 MWh / 1 MW LFP BESS in SE3
   CAPEX: €560,000  |  Discount: 8%  |  Warranty: 6000 cycles, 15 yrs
   Fixed O&M: €8000/MW/year

 c_deg  annual_revenue_net  annual_cycles  life_years  annual_cashflow  lifetime_revenue       npv  payback_yrs
     0             67689.0          796.0         8.0          59689.0          449844.0 -231938.0          9.0
     5             57627.0          493.0        12.0          49627.0          604080.0 -182861.0         11.0
    10             50022.0          416.0        14.0          42022.0          605602.0 -208109.0         13.0
    20             38663.0          302.0        15.0          30663.0          459940.0 -297544.0         18.0
    50             21758.0          148.0        15.0          13758.0          206375.0 -442236.0         41.0



🏆 NPV-optimal operating point:
   Degradation cost: €5/MWh throughput
   Annual revenue:   €57,627/MW/year
   Annual cycles:    493
   Battery life:     12.2 years
   Lifetime NPV:     €-182,861
   Payback period:   11.3 years


In [5]:
# At what CAPEX/MWh does DA-only arbitrage break even?
best_cashflow = sens_df.loc[sens_df["npv"].idxmax(), "annual_cashflow"]
best_life = sens_df.loc[sens_df["npv"].idxmax(), "life_years"]
r = 0.08

# PV of cashflow stream
pv_factor = sum(1 / (1 + r)**t for t in range(1, int(best_life) + 1))
breakeven_capex = best_cashflow * pv_factor

print(f"Best-case annual cashflow: €{best_cashflow:,.0f}")
print(f"Best-case battery life:    {best_life:.1f} years")
print(f"Discount factor sum:       {pv_factor:.2f}")
print(f"\n💰 Break-even total CAPEX: €{breakeven_capex:,.0f}")
print(f"   Break-even CAPEX/MWh:   €{breakeven_capex/2:,.0f}/MWh")
print(f"   Current CAPEX/MWh:      €280,000/MWh")
print(f"   Gap:                    €{280000 - breakeven_capex/2:,.0f}/MWh ({100*(280000 - breakeven_capex/2)/280000:.0f}% reduction needed)")
print(f"\nAt 10%/year CAPEX decline, that's {np.log(breakeven_capex / (560000)) / np.log(0.9):.1f} years away")

Best-case annual cashflow: €49,627
Best-case battery life:    12.2 years
Discount factor sum:       7.54

💰 Break-even total CAPEX: €373,994
   Break-even CAPEX/MWh:   €186,997/MWh
   Current CAPEX/MWh:      €280,000/MWh
   Gap:                    €93,003/MWh (33% reduction needed)

At 10%/year CAPEX decline, that's 3.8 years away


In [6]:
import requests
import pandas as pd

lat, lon = 59.33, 18.06  # Stockholm/SE3

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": lat,
    "longitude": lon,
    "start_date": "2022-11-01",
    "end_date": "2026-06-04",
    "hourly": ",".join([
        "temperature_2m",
        "relative_humidity_2m",
        "wind_speed_10m",
        "wind_speed_100m",
        "wind_direction_100m",
        "shortwave_radiation",
        "cloud_cover",
        "pressure_msl",
        "precipitation",        # mm/h - hydro inflow proxy
        "snowfall",             # cm/h - delayed inflow (snowmelt in spring)
    ]),
    "timezone": "UTC",
}

print("Fetching weather + precipitation from Open-Meteo...")
r = requests.get(url, params=params, timeout=60)
data = r.json()

weather_df = pd.DataFrame(data["hourly"])
weather_df["time"] = pd.to_datetime(weather_df["time"], utc=True)
weather_df = weather_df.rename(columns={"time": "timestamp"})

print(f"\n✅ Fetched {len(weather_df):,} hours")
print(f"📅 Range: {weather_df['timestamp'].min()} → {weather_df['timestamp'].max()}")
print(f"\nKey stats:")
print(f"  Temperature (°C):   {weather_df['temperature_2m'].min():.1f} → {weather_df['temperature_2m'].max():.1f}")
print(f"  Wind 100m (m/s):    mean {weather_df['wind_speed_100m'].mean():.1f}")
print(f"  Solar (W/m²):       mean {weather_df['shortwave_radiation'].mean():.0f}")
print(f"  Precipitation:      total {weather_df['precipitation'].sum():.0f} mm over period")

weather_df.to_parquet(f"{PROCESSED_PATH}/se3_weather_2022_2026.parquet")
print(f"\n💾 Saved to {PROCESSED_PATH}/se3_weather_2022_2026.parquet")

Fetching weather + precipitation from Open-Meteo...

✅ Fetched 31,488 hours
📅 Range: 2022-11-01 00:00:00+00:00 → 2026-06-04 23:00:00+00:00

Key stats:
  Temperature (°C):   -18.2 → 27.9
  Wind 100m (m/s):    mean 21.6
  Solar (W/m²):       mean 114
  Precipitation:      total 2232 mm over period

💾 Saved to /content/drive/MyDrive/bess-optimizer-sweden/data/processed/se3_weather_2022_2026.parquet


In [7]:
import numpy as np

w = pd.read_parquet(f"{PROCESSED_PATH}/se3_weather_2022_2026.parquet")

# Aggregate to daily
daily_w = w.set_index("timestamp").resample("1D").agg({
    "temperature_2m": "mean",
    "wind_speed_100m": "mean",
    "shortwave_radiation": "mean",
    "precipitation": "sum",
    "snowfall": "sum",
}).reset_index()

# Hydro reservoir proxies — cumulative precipitation over various windows
# (Norwegian/Swedish reservoirs fill from rainfall and spring snowmelt with multi-week lag)
daily_w["precip_30d"]   = daily_w["precipitation"].rolling(30,  min_periods=1).sum()
daily_w["precip_90d"]   = daily_w["precipitation"].rolling(90,  min_periods=1).sum()
daily_w["precip_180d"]  = daily_w["precipitation"].rolling(180, min_periods=1).sum()
daily_w["snow_60d"]     = daily_w["snowfall"].rolling(60, min_periods=1).sum()

# "Reservoir proxy" — combined long-window precipitation
# High value = reservoirs filling = lower future prices expected (hydro abundance)
daily_w["reservoir_proxy"] = (
    daily_w["precip_180d"] / daily_w["precip_180d"].rolling(365, min_periods=30).mean()
)

daily_w.to_parquet(f"{PROCESSED_PATH}/se3_daily_weather_hydro.parquet")

print("✅ Hydro proxy features computed:")
print(f"   30-day precip:   mean {daily_w['precip_30d'].mean():.0f} mm  (range {daily_w['precip_30d'].min():.0f} - {daily_w['precip_30d'].max():.0f})")
print(f"   180-day precip:  mean {daily_w['precip_180d'].mean():.0f} mm")
print(f"   Reservoir proxy: range {daily_w['reservoir_proxy'].min():.2f} - {daily_w['reservoir_proxy'].max():.2f}  (1.0 = long-term average)")
print(f"\n💾 Saved to {PROCESSED_PATH}/se3_daily_weather_hydro.parquet")
print(f"   ({len(daily_w)} daily rows × {len(daily_w.columns)} columns)")

print("\n📝 Note: This is a precipitation-based proxy. For Phase 5 we may replace with")
print("   actual Nordic reservoir data from Nord Pool (manual download) or ENTSO-E.")

✅ Hydro proxy features computed:
   30-day precip:   mean 51 mm  (range 2 - 164)
   180-day precip:  mean 293 mm
   Reservoir proxy: range 0.47 - 2.44  (1.0 = long-term average)

💾 Saved to /content/drive/MyDrive/bess-optimizer-sweden/data/processed/se3_daily_weather_hydro.parquet
   (1312 daily rows × 11 columns)

📝 Note: This is a precipitation-based proxy. For Phase 5 we may replace with
   actual Nordic reservoir data from Nord Pool (manual download) or ENTSO-E.


In [8]:
df_prices = pd.read_parquet(f"{PROCESSED_PATH}/se3_hourly_2022_2026.parquet")

# Day-ahead market mechanics (Nord Pool):
#   - Gate closure: 12:00 CET on D-1
#   - Market clearing + publication: ~13:00 CET on D-1
#   - Delivery: hours of day D
#
# So for any delivery_time on day D, the price became "known" at ~12:00 UTC on D-1
# (assuming CET = UTC+1, ignoring DST for simplicity; the timestamp is already UTC)

df_prices["delivery_date"] = pd.to_datetime(df_prices["timestamp"]).dt.date
df_prices["knowledge_time"] = (
    pd.to_datetime(df_prices["delivery_date"]) - pd.Timedelta(days=1)
).dt.tz_localize("UTC") + pd.Timedelta(hours=12)

# Save augmented version
df_prices.to_parquet(f"{PROCESSED_PATH}/se3_hourly_2022_2026_v2.parquet")

# Verify
sample = df_prices.iloc[[0, 100, 1000, 10000, -1]]
print("✅ knowledge_time column added")
print(f"\nSample rows showing delivery_time vs knowledge_time:")
print(sample[["timestamp", "knowledge_time", "price_eur_mwh"]].to_string(index=False))

# Compute the gate-to-delivery lead time
lead = (pd.to_datetime(df_prices["timestamp"]) - df_prices["knowledge_time"])
print(f"\nLead time (delivery − gate publication):")
print(f"  Min:    {lead.min()}")
print(f"  Max:    {lead.max()}")
print(f"  Mean:   {lead.mean()}")
print(f"\n💾 Saved to {PROCESSED_PATH}/se3_hourly_2022_2026_v2.parquet")

print("\n📌 Why this matters: when we run rolling-horizon MPC in Phase 5, we'll filter")
print("   the dataset by 'knowledge_time <= decision_time' to prevent lookahead bias.")
print("   This methodology discipline is what separates 'student backtest' from 'research-grade'.")

✅ knowledge_time column added

Sample rows showing delivery_time vs knowledge_time:
                timestamp            knowledge_time  price_eur_mwh
2022-10-31 23:00:00+00:00 2022-10-30 12:00:00+00:00        34.9100
2022-11-05 03:00:00+00:00 2022-11-04 12:00:00+00:00        23.4300
2022-12-12 15:00:00+00:00 2022-12-11 12:00:00+00:00       542.3500
2023-12-22 15:00:00+00:00 2023-12-21 12:00:00+00:00        41.3900
2026-06-04 21:00:00+00:00 2026-06-03 12:00:00+00:00        59.0725

Lead time (delivery − gate publication):
  Min:    0 days 12:00:00
  Max:    1 days 11:00:00
  Mean:   0 days 23:29:58.799504557

💾 Saved to /content/drive/MyDrive/bess-optimizer-sweden/data/processed/se3_hourly_2022_2026_v2.parquet

📌 Why this matters: when we run rolling-horizon MPC in Phase 5, we'll filter
   the dataset by 'knowledge_time <= decision_time' to prevent lookahead bias.
   This methodology discipline is what separates 'student backtest' from 'research-grade'.
